# Orientation test (local CSV)

This notebook loads the provided inverter CSV, runs the Solar Data Tools pipeline, and estimates tilt/azimuth.
Update `DATA_PATH` and `METADATA_PATH` if you store the files elsewhere.

In [ ]:
from pathlib import Path
import json
import pandas as pd
from solardatatools import DataHandler

DATA_PATH = Path(r"C:\Users\Maren.Murjahn\Nextcloud2\PRO-Projektarbeit-2025\pv_analyser\data\sonnja_pv3_2015\einleuchtend_wrdata_2015_wr1.csv")
METADATA_PATH = Path(r"C:\Users\Maren.Murjahn\Nextcloud2\PRO-Projektarbeit-2025\pv_analyser\data\sonnja_pv3_2015\metadata.json")

df = pd.read_csv(DATA_PATH, sep=';', decimal='.', parse_dates=['timestamp'])
df = df.sort_values('timestamp').set_index('timestamp')

with METADATA_PATH.open('r', encoding='utf-8') as f:
    meta = json.load(f)

df.head()

In [ ]:
# Use AC power if available; fall back to DC power if needed
power_col = 'P_AC' if 'P_AC' in df.columns else 'P_DC'

data = pd.DataFrame({
    'power': pd.to_numeric(df[power_col], errors='coerce')
}, index=df.index)

data = data.dropna(subset=['power'])
data.head()

In [ ]:
dh = DataHandler(data)
dh.run_pipeline()

# GMT offset for Europe/Berlin is typically +1 (winter) or +2 (summer).
# Use +1 for a simple first test.
dh.setup_location_and_orientation_estimation(gmt_offset=1)

tilt_est, az_est = dh.estimate_orientation()
print(f"Estimated tilt: {tilt_est:.2f}")
print(f"Estimated azimuth: {az_est:.2f}")

In [ ]:
# Compare against metadata if available
true_tilt = meta.get('true_tilt')
true_az = meta.get('true_azimuth')

if true_tilt is not None and true_az is not None:
    az_error = (true_az - az_est + 180) % 360 - 180
    print(f"True tilt: {true_tilt:.2f}")
    print(f"Tilt error: {true_tilt - tilt_est:.2f}")
    print('---')
    print(f"True azimuth: {true_az:.2f}")
    print(f"Azimuth error: {az_error:.2f}")